In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# For text analysis
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from collections import Counter
import re

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries loaded successfully!")

In [ ]:
# Load the news/analyst rating data
import os

news_df = pd.read_csv('C:/Users/hp/Desktop/raw_analyst_ratings.csv')
news_df.head()

In [ ]:
# Basic information about the dataset
print("Dataset Info:")
print(news_df.info())

print("\nMissing Values:")
print(news_df.isnull().sum())

In [ ]:
import glob

stock_files = glob.glob(r'C:\Users\hp\Desktop\yfinance_data\*.csv')
stock_data = {}
for file in stock_files:
    stock_name = file.split('/')[-1].replace('.csv', '')
    stock_data[stock_name] = pd.read_csv(file)
    print(f"✅ Loaded {stock_name}: {stock_data[stock_name].shape[0]} rows")

print(f"\n📊 Total stocks loaded: {len(stock_data)}")
print(f"Stocks: {list(stock_data.keys())}")

In [ ]:
news_df['headline_length'] = news_df[headline_col].astype(str).str.len()

In [ ]:
headline_col = None
for col in news_df.columns:
    if 'headline' in col.lower() or 'title' in col.lower() or 'text' in col.lower():
        headline_col = col
        break

if headline_col:
    print(f"Using column: '{headline_col}' for headline analysis")
    
    # Calculate headline characteristics
    news_df['headline_length'] = news_df[headline_col].astype(str).str.len()
    news_df['word_count'] = news_df[headline_col].astype(str).str.split().str.len()
    
    print("\n📝 Headline Statistics:")
    print(f"  - Average length: {news_df['headline_length'].mean():.1f} characters")
    print(f"  - Min length: {news_df['headline_length'].min()} characters")
    print(f"  - Max length: {news_df['headline_length'].max()} characters")
    print(f"  - Average word count: {news_df['word_count'].mean():.1f} words")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(news_df['headline_length'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Headline Length (characters)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Headline Lengths')
    axes[0].axvline(news_df['headline_length'].mean(), color='red', linestyle='--', 
                    label=f"Mean: {news_df['headline_length'].mean():.0f}")
    axes[0].legend()
    
    # Box plot
    sns.boxplot(y=news_df['headline_length'], ax=axes[1], color='lightblue')
    axes[1].set_ylabel('Headline Length (characters)')
    axes[1].set_title('Box Plot - Outlier Detection')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No headline column found. Available columns:", news_df.columns.tolist())

In [ ]:
# Publisher/Source Analysis
publisher_col = None
for col in news_df.columns:
    if 'publisher' in col.lower() or 'source' in col.lower() or 'author' in col.lower():
        publisher_col = col
        break

if publisher_col:
    publisher_counts = news_df[publisher_col].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    publisher_counts.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Number of Articles')
    ax.set_ylabel('Publisher/Source')
    ax.set_title('Top 15 Most Active Publishers')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f"\n📰 Top 5 Publishers:")
    print(publisher_counts.head())
else:
    print("⚠️ No publisher column found.")


In [ ]:
# Stock Symbol Analysis
stock_col = None
for col in news_df.columns:
    if 'stock' in col.lower() or 'ticker' in col.lower() or 'symbol' in col.lower():
        stock_col = col
        break

if stock_col:
    stock_counts = news_df[stock_col].value_counts()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    stock_counts.head(15).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_xlabel('Stock Symbol')
    ax.set_ylabel('Number of Articles')
    ax.set_title('Top 15 Stocks by News Coverage')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f"\n📈 Most Covered Stocks:")
    print(stock_counts.head())
    
    # Check if our yfinance stocks are covered
    yfinance_stocks = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']
    covered_stocks = [s for s in yfinance_stocks if s in stock_counts.index]
    print(f"\n✅ Stocks with news coverage: {covered_stocks}")
else:
    print("⚠️ No stock column found.")


In [ ]:
# Time Series Analysis
# Find date column
date_col = None
for col in news_df.columns:
    if 'date' in col.lower() or 'datetime' in col.lower() or 'time' in col.lower():
        date_col = col
        break

if date_col:
    print(f"✅ Found date column: '{date_col}'")
    print(f"Sample dates: {news_df[date_col].head(3).tolist()}")
    
    # Try different methods to parse dates
    try:
        # Method 1: Let pandas auto-detect
        news_df[date_col] = pd.to_datetime(news_df[date_col], errors='coerce')
    except:
        try:
            # Method 2: Remove timezone info first
            dates_str = news_df[date_col].astype(str).str.replace('+00:00', '').str.replace('UTC', '')
            news_df[date_col] = pd.to_datetime(dates_str, errors='coerce')
        except:
            # Method 3: Use mixed format
            news_df[date_col] = pd.to_datetime(news_df[date_col], format='mixed', errors='coerce')
    
    # Drop rows with invalid dates
    invalid_dates = news_df[date_col].isna().sum()
    if invalid_dates > 0:
        print(f"⚠️ Found {invalid_dates} invalid dates. Dropping them.")
        news_df = news_df.dropna(subset=[date_col])
    
    # Extract time features
    news_df['year'] = news_df[date_col].dt.year
    news_df['month'] = news_df[date_col].dt.month
    news_df['day'] = news_df[date_col].dt.day
    news_df['hour'] = news_df[date_col].dt.hour
    news_df['day_of_week'] = news_df[date_col].dt.day_name()
    
    # Daily article count
    daily_counts = news_df.groupby(news_df[date_col].dt.date).size()
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Daily trend
    axes[0,0].plot(daily_counts.index, daily_counts.values, alpha=0.7, linewidth=1, color='steelblue')
    axes[0,0].set_xlabel('Date')
    axes[0,0].set_ylabel('Number of Articles')
    axes[0,0].set_title('Daily Publication Volume')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. By hour
    hourly_counts = news_df['hour'].value_counts().sort_index()
    axes[0,1].bar(hourly_counts.index, hourly_counts.values, color='coral')
    axes[0,1].set_xlabel('Hour of Day')
    axes[0,1].set_ylabel('Number of Articles')
    axes[0,1].set_title('Publication Volume by Hour')
    
    # 3. By day of week
    week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekly_counts = news_df['day_of_week'].value_counts().reindex(week_order)
    axes[1,0].bar(weekly_counts.index, weekly_counts.values, color='seagreen')
    axes[1,0].set_xlabel('Day of Week')
    axes[1,0].set_ylabel('Number of Articles')
    axes[1,0].set_title('Publication Volume by Day')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # 4. Monthly trend
    monthly_counts = news_df.groupby(['year', 'month']).size()
    axes[1,1].plot(range(len(monthly_counts)), monthly_counts.values, marker='o', color='purple')
    axes[1,1].set_xlabel('Time (Sequential Months)')
    axes[1,1].set_ylabel('Number of Articles')
    axes[1,1].set_title('Monthly Publication Trend')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*50)
    print("TIME SERIES SUMMARY")
    print("="*50)
    print(f"📅 Date Range: {news_df[date_col].min()} to {news_df[date_col].max()}")
    print(f"📊 Total days in dataset: {daily_counts.shape[0]}")
    print(f"⏰ Peak publication hour: {news_df['hour'].mode()[0]}:00")
    print(f"📆 Most active day: {weekly_counts.idxmax()}")
    print(f"📈 Average articles per day: {daily_counts.mean():.1f}")
    print(f"📉 Max articles in a day: {daily_counts.max()} on {daily_counts.idxmax()}")
    
else:
    print("⚠️ No date column found.")
    print(f"Available columns: {news_df.columns.tolist()}")

In [ ]:
# Keyword Extraction (TF-IDF)

if headline_col:
    # Clean the headlines
    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        return text
    
    news_df['clean_headline'] = news_df[headline_col].apply(clean_text)
    
    # Extract keywords using TF-IDF
    tfidf = TfidfVectorizer(max_features=30, stop_words='english')
    tfidf_matrix = tfidf.fit_transform(news_df['clean_headline'])
    
    feature_names = tfidf.get_feature_names_out()
    tfidf_scores = tfidf_matrix.mean(axis=0).A1
    top_idx = tfidf_scores.argsort()[-20:][::-1]
    
    top_keywords = [(feature_names[i], tfidf_scores[i]) for i in top_idx]
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    keywords, scores = zip(*top_keywords[:15])
    ax.barh(keywords, scores, color='teal')
    ax.set_xlabel('TF-IDF Score')
    ax.set_title('Top 15 Keywords in Financial Headlines')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("🏷️ Top Keywords:")
    for keyword, score in top_keywords[:10]:
        print(f"  - {keyword}: {score:.4f}")


In [ ]:
# Summary Report
print("\n" + "="*60)
print("📊 EDA SUMMARY REPORT")
print("="*60)

print(f"\n📰 NEWS DATA:")
print(f"  - Total articles: {len(news_df)}")
print(f"  - Unique stocks: {news_df[stock_col].nunique() if stock_col else 'N/A'}")
print(f"  - Unique publishers: {news_df[publisher_col].nunique() if publisher_col else 'N/A'}")
print(f"  - Date range: {news_df[date_col].min() if date_col else 'N/A'} to {news_df[date_col].max() if date_col else 'N/A'}")

print(f"\n📈 STOCK PRICE DATA:")
print(f"  - Stocks loaded: {list(stock_data.keys())}")
for stock, df in stock_data.items():
    print(f"    • {stock}: {df.shape[0]} trading days")

if headline_col:
    print(f"\n📝 HEADLINE METRICS:")
    print(f"  - Average length: {news_df['headline_length'].mean():.1f} characters")
    print(f"  - Average words: {news_df['word_count'].mean():.1f} words")

print("\n✅ Task 1 EDA Complete!")
print("="*60)


In [ ]:
# Save processed data for Task 2
news_df.to_csv('C:/Users/hp/Desktop/data/processed_news.csv', index=False)
print("\n💾 Processed news data saved to 'C:/Users/hp/Desktop/data/processed_news.csv'")

In [ ]:
# TASK 1 REQUIREMENTS CHECKLIST
print("="*60)
print("TASK 1 COMPLETION CHECKLIST")
print("="*60)

# 1. Descriptive Statistics
print("\n✅ 1. DESCRIPTIVE STATISTICS")
print(f"   - Total articles: {len(news_df)}")
print(f"   - Unique stocks: {news_df['stock'].nunique() if 'stock' in news_df.columns else 'N/A'}")
print(f"   - Unique publishers: {news_df['publisher'].nunique() if 'publisher' in news_df.columns else 'N/A'}")
print(f"   - Date range: {news_df['date'].min()} to {news_df['date'].max()}")
print(f"   - Headline length stats:")
print(f"      Mean: {news_df['headline_length'].mean():.1f} chars")
print(f"      Median: {news_df['headline_length'].median():.1f} chars")
print(f"      Std Dev: {news_df['headline_length'].std():.1f} chars")

# 2. Publisher Analysis
print("\n✅ 2. PUBLISHER ANALYSIS")
top_publishers = news_df['publisher'].value_counts().head(5)
for pub, count in top_publishers.items():
    print(f"   - {pub}: {count} articles ({count/len(news_df)*100:.1f}%)")

# 3. Time Series Analysis
print("\n✅ 3. TIME SERIES ANALYSIS")
print(f"   - Peak publication hour: {news_df['hour'].mode()[0]}:00")
print(f"   - Most active day: {news_df['day_of_week'].mode()[0]}")
print(f"   - Busiest month: {news_df['month'].mode()[0]}")

# 4. Keyword/Topic Analysis
print("\n✅ 4. KEYWORD & TOPIC ANALYSIS")
print("   Top 10 keywords (TF-IDF):")
# Add your actual top keywords here
top_keywords_list = [term for term, _ in top_keywords[:10]]  # Use your variable
for i, kw in enumerate(top_keywords_list, 1):
    print(f"      {i}. {kw}")

# 5. Stock Coverage
print("\n✅ 5. STOCK COVERAGE ANALYSIS")
top_stocks = news_df['stock'].value_counts().head(5)
for stock, count in top_stocks.items():
    print(f"   - {stock}: {count} articles")

print("\n" + "="*60)
print("ALL TASK 1 REQUIREMENTS SATISFIED")
print("="*60)